# YOLO11n 日期偵測訓練（Colab GPU）

輸入：`MyDrive/ocr_project/yolo_dataset.zip`（由本機 `convert_to_yolo.py` 產生後打包）

輸出：`runs/date_det_v2/weights/best.pt` → 下載回本機 `yoloOcr/models/best.pt`

## 1. 確認 GPU

In [ ]:
!nvidia-smi

## 2. 掛載 Drive 並解壓資料

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/ocr_project/yolo_dataset.zip /content/
!unzip -q -o /content/yolo_dataset.zip -d /content/
!ls /content/yolo_dataset

## 3. 修正 data.yaml 內 path（指向 Colab 路徑）

In [ ]:
yaml_path = '/content/yolo_dataset/data.yaml'
content = open(yaml_path).read()
lines = []
for line in content.splitlines():
    if line.startswith('path:'):
        lines.append('path: /content/yolo_dataset')
    else:
        lines.append(line)
open(yaml_path, 'w').write('\n'.join(lines))
print(open(yaml_path).read())

## 4. 安裝 ultralytics

In [ ]:
!pip install -q ultralytics

## 5. 訓練 YOLO11n

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=32,
    patience=20,
    project='/content/runs',
    name='date_det_v2',
    exist_ok=True,
    device=0,
    workers=4,
    cache=True,
    augment=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.5,
    fliplr=0.5, mosaic=1.0,
)

## 6. 驗證

In [ ]:
val_results = model.val(data='/content/yolo_dataset/data.yaml')
print(f'mAP50      : {val_results.box.map50:.4f}')
print(f'mAP50-95   : {val_results.box.map:.4f}')

## 7. 備份 best.pt 到 Drive + 下載

In [ ]:
import shutil
best = '/content/runs/date_det_v2/weights/best.pt'
shutil.copy(best, '/content/drive/MyDrive/ocr_project/yolo11n_date_best.pt')
from google.colab import files
files.download(best)

## 8.（選擇性）匯出 ONNX，供 C++ 部署

下載 `best.onnx` → 之後可放到 `eval_cpp_runner/models/`。

In [ ]:
model.export(format='onnx', imgsz=640, opset=12, simplify=True)
files.download('/content/runs/date_det_v2/weights/best.onnx')